# Strands Agent with AgentCore Deployment

This notebook demonstrates how to build a ReAct-style agent using the **Strands Agents SDK** and **AWS Bedrock**, then deploy it as a managed service on **Amazon Bedrock AgentCore Runtime**.

You'll learn how to:
- Configure and connect to Claude models via Bedrock
- Define custom tools that the agent can use
- Build an agent that reasons and acts autonomously
- Test the agent with various queries
- Package and deploy the agent to AgentCore Runtime
- Invoke the deployed agent via CLI and boto3

**Prerequisites:** Ensure your model is configured in `../CONFIG.txt` before running.

## 1. Setup

Check which packages are pre-installed, then install any missing dependencies. The key packages are:
- **strands-agents**: The Strands Agents SDK for building AI agents
- **strands-agents-tools**: Pre-built tools for common tasks

In [ ]:
import importlib.metadata

packages = [
    "strands-agents",
    "strands-agents-tools",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
%pip install strands-agents strands-agents-tools -q

## 2. Imports

Import the required libraries:
- **strands**: `Agent` class and `tool` decorator for defining agent tools
- **strands.models**: `BedrockModel` for connecting to Claude on Amazon Bedrock

In [ ]:
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel

print("Imports successful!")

## 3. Configuration

Load the model ID and AWS region from `CONFIG.txt`. For cross-region inference profiles (MODEL_ID starting with `us.`), Bedrock routes requests to the optimal region automatically.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = os.getenv("REGION", "us-east-1")

print(f"Model ID: {MODEL_ID}")
print(f"Region:   {REGION}")

## 4. Define Tools

Tools are functions that the agent can call to perform actions or retrieve information. We use the `@tool` decorator from Strands to register them.

The agent will automatically decide when to use these tools based on the user's question.

In [ ]:
@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


print(f"Defined tools: get_current_time, add_numbers")

## 5. Create the Agent

Create a Strands `Agent` with:
- **BedrockModel**: Connects to Claude via Amazon Bedrock with temperature 0 for deterministic responses
- **Tools**: The custom tools we defined above
- **System prompt**: Instructions for the agent's behavior

Unlike graph-based frameworks that require defining nodes, edges, and routing logic, Strands handles the ReAct loop (Reason → Act → Observe → Repeat) internally. The agent decides when to use tools and when to respond directly.

In [ ]:
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

agent = Agent(
    model=bedrock_model,
    system_prompt="You are a helpful assistant. Use the available tools when needed to answer questions.",
    tools=[get_current_time, add_numbers],
)

print(f"Agent created with model: {MODEL_ID}")

## 6. Run the Agent

The `run_agent` function sends a question to the agent and displays the response. The agent will automatically reason about whether to use tools, call them if needed, and synthesize a final answer.

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"Question: {question}")
    print("-" * 50)
    response = agent(question)
    print(f"\nResponse:\n{response}")
    return response

In [ ]:
# Test: Get current time
result = run_agent("What is the current time?")

In [ ]:
# Test: Math calculation
result = run_agent("What is 42 + 17?")

In [ ]:
# Test: Multiple tools
result = run_agent("What time is it and what is 100 + 200?")

## 7. Query with Sample Financial Data

This section demonstrates how to provide context to the agent by loading sample SEC 10-K filing data and asking questions about it.

This pattern is useful for:
- **RAG (Retrieval-Augmented Generation)**: Injecting retrieved documents into prompts
- **In-context learning**: Providing examples or data for the agent to reference
- **Domain-specific Q&A**: Answering questions based on financial filing information

In [ ]:
from load_sample_data import load_financial_data, print_info

financial_text = load_financial_data()
print_info(financial_text)

In [ ]:
def ask_about_data(question: str, context: str):
    """Ask the agent a question with context."""
    prompt = f"""Based on this SEC 10-K filing information:

{context}

Question: {question}"""
    return run_agent(prompt)

# Sample question about companies and products
ask_about_data("What companies are mentioned and what are their key products?", financial_text)

In [ ]:
# Another sample question about risk factors
ask_about_data("What risk factors are mentioned and which companies face them?", financial_text)

---

## 8. Deploy to AgentCore Runtime

Now that the agent works locally, let's deploy it as a **managed service** on Amazon Bedrock AgentCore Runtime. This packages the agent code and deploys it using the AgentCore starter toolkit — no Docker required.

**Architecture:**
```
User → AgentCore Runtime → Strands Agent (Claude) → Tools
```

The deployment process:
1. Create a self-contained directory with the agent code and dependencies
2. Use the `agentcore` CLI to deploy via CodeBuild
3. Invoke the deployed agent via CLI or boto3

In [ ]:
%pip install bedrock-agentcore-starter-toolkit>=0.3.3 bedrock-agentcore>=1.4.7 pyyaml -q

## 9. Create the Deployment Package

AgentCore packages an entire directory and uploads it. We create a self-contained directory with:
- `agent.py` — the agent code wrapped in a `BedrockAgentCoreApp` handler
- `pyproject.toml` — Python dependencies

In [ ]:
!mkdir -p agentcore_deploy
print("Created agentcore_deploy/ directory")

In [ ]:
%%writefile agentcore_deploy/agent.py
#!/usr/bin/env python3
"""
Basic Strands Agent - AgentCore Runtime Deployment

A simple agent with time and math tools, deployed to AgentCore Runtime.
"""

import logging
import os
from datetime import datetime

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-20250514-v1:0")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

SYSTEM_PROMPT = "You are a helpful assistant. Use the available tools when needed to answer questions."


@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@app.entrypoint
async def invoke(payload: dict = None):
    """AgentCore Runtime handler."""
    if payload is None:
        payload = {}

    prompt = (
        payload.get("prompt")
        or payload.get("message")
        or payload.get("query")
        or payload.get("input")
    )

    if not prompt:
        yield {"type": "error", "error": "No prompt provided. Include 'prompt' in your request."}
        return

    logger.info(f"Query: {prompt[:100]}...")

    try:
        model = BedrockModel(
            model_id=MODEL_ID,
            region_name=AWS_REGION,
            temperature=0,
        )
        agent = Agent(
            model=model,
            system_prompt=SYSTEM_PROMPT,
            tools=[get_current_time, add_numbers],
        )

        response = agent(prompt)
        response_text = str(response)

        yield {"type": "chunk", "data": response_text}
        yield {"type": "complete"}

        logger.info("Request completed successfully")

    except Exception as e:
        logger.error(f"Error: {e}", exc_info=True)
        yield {"type": "error", "error": f"Error processing request: {str(e)}"}


if __name__ == "__main__":
    app.run(port=8080)

In [ ]:
%%writefile agentcore_deploy/pyproject.toml
[project]
name = "basic-strands-agent"
version = "0.1.0"
description = "Basic Strands agent with time and math tools"
requires-python = ">=3.10"
dependencies = [
    "strands-agents>=0.1.0",
    "strands-agents-tools>=0.1.0",
    "boto3>=1.42.0",
    "bedrock-agentcore>=1.4.7",
]

In [ ]:
!ls -la agentcore_deploy/

## 10. Configure and Deploy

AgentCore needs a `.bedrock_agentcore.yaml` config file that specifies the entrypoint, deployment type, and AWS settings. We generate this programmatically to avoid interactive CLI prompts.

The deployment uses CodeBuild to build an ARM64 container — no Docker is required locally.

In [ ]:
import boto3
import yaml

sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

agent_dir = os.path.abspath("agentcore_deploy")
entrypoint = os.path.join(agent_dir, "agent.py")

AGENT_NAME = "basic_strands_agent"

config = {
    "default_agent": AGENT_NAME,
    "agents": {
        AGENT_NAME: {
            "name": AGENT_NAME,
            "language": "python",
            "entrypoint": entrypoint,
            "deployment_type": "direct_code_deploy",
            "runtime_type": "PYTHON_3_13",
            "platform": "linux/arm64",
            "source_path": agent_dir,
            "aws": {
                "account": account_id,
                "region": REGION,
                "execution_role_auto_create": True,
                "ecr_auto_create": False,
                "s3_auto_create": True,
                "network_configuration": {
                    "network_mode": "PUBLIC",
                },
                "protocol_configuration": {
                    "server_protocol": "HTTP",
                },
                "observability": {
                    "enabled": True,
                },
            },
        }
    },
}

config_path = os.path.join(agent_dir, ".bedrock_agentcore.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {config_path}")
print(f"  Agent:      {AGENT_NAME}")
print(f"  Account:    {account_id}")
print(f"  Region:     {REGION}")
print(f"  Deploy:     direct_code_deploy")

In [ ]:
# Install zip if not available (needed by direct_code_deploy)
!which zip || (sudo apt-get update -qq && sudo apt-get install -y -qq zip)

# Deploy (--auto-update-on-conflict updates the agent if it already exists)
!cd agentcore_deploy && agentcore deploy --auto-update-on-conflict

## 11. Invoke the Deployed Agent

Once deployed, invoke the agent via the `agentcore` CLI or programmatically with boto3.

In [ ]:
# Invoke via CLI
!cd agentcore_deploy && agentcore invoke '{"prompt": "What time is it and what is 42 + 17?"}'

### Invoke via boto3

For programmatic access, use the `bedrock-agentcore` boto3 client:

In [ ]:
import json
import uuid
import yaml
from botocore.config import Config

with open("agentcore_deploy/.bedrock_agentcore.yaml") as f:
    deploy_config = yaml.safe_load(f)

default_agent = deploy_config["default_agent"]
agent_config = deploy_config["agents"][default_agent]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]
AGENT_REGION = agent_config["aws"]["region"]

print(f"Agent:  {default_agent}")
print(f"ARN:    {AGENT_ARN}")
print(f"Region: {AGENT_REGION}")

In [ ]:
agentcore_config = Config(read_timeout=300)


def invoke_agent(prompt: str):
    """Invoke the deployed agent via boto3."""
    client = boto3.client(
        "bedrock-agentcore",
        region_name=AGENT_REGION,
        config=agentcore_config,
    )

    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )

    content = "".join(
        chunk.decode("utf-8") for chunk in response.get("response", [])
    )

    try:
        return json.loads(content)
    except (json.JSONDecodeError, ValueError):
        return content


result = invoke_agent("What is 100 + 200?")
print(result)

## 12. Cleanup

When you're done, remove the agent from AgentCore Runtime. This deletes the deployed runtime but keeps your local files.

In [ ]:
# Uncomment the line below to destroy the deployed agent
# !cd agentcore_deploy && agentcore destroy